In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
# 5. RoPE：对 Q/K 的偶数-奇数维做二维旋转
def rotate_half(x):
    x_even, x_odd = x[..., ::2], x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)

def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    assert head_dim % 2 == 0
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    angles = torch.outer(torch.arange(seq_len, device=device).float(), inv_freq)
    angles = torch.repeat_interleave(angles, 2, dim=-1)[None, None]  # [1,1,T,Dh]
    return angles.cos(), angles.sin()

def apply_rope(q, k, cos, sin):
    assert q.shape[-2:] == k.shape[-2:] == cos.shape[-2:]
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin

q = torch.randn(2, 4, 5, 8, device=device)
k = torch.randn(2, 4, 5, 8, device=device)
cos, sin = build_rope_cache(5, 8, device)
q_rope, k_rope = apply_rope(q, k, cos, sin)
assert torch.allclose(q.norm(dim=-1), q_rope.norm(dim=-1), atol=1e-5)